# Question 2 - Classement par le modèle Bradley-Terry

compare-ia vote
Modèle de Bradley-Terry

## 0. Mise en place et exploration des données préliminaires

In [ ]:
import pandas as pd
import numpy as np

df_votes = pd.read_parquet(r"C:\Users\Asturiel\Documents\Cours\CentraleSupelec_COURS\Third_Year\Challenge-etude_de_cas-creativite\data\compareia-votes\votes.parquet")
display(df_votes)

,id,timestamp,model_a_name,model_b_name,model_pair_name,chosen_model_name,opening_msg,both_equal,conversation_a,conversation_b,...,conv_incorrect_a,conv_incorrect_b,conv_superficial_a,conv_superficial_b,conv_instructions_not_followed_a,conv_instructions_not_followed_b,system_prompt_b,system_prompt_a,conv_complete_a,conv_complete_b
0,112580,2025-10-28 17:02:44.266579,gemini-2.5-flash,grok-4-fast,"[gemini-2.5-flash, grok-4-fast]",None,crée des cartes types dixit sur le thème des m...,1.0,None,None,...,False,False,False,False,False,False,None,None,False,False
1,55251,2025-04-22 18:07:44.914551,gemma-3-4b,c4ai-command-r-08-2024,"[c4ai-command-r-08-2024, gemma-3-4b]",gemma-3-4b,J'ai lu quelque part (je ne sais pas où) qu'ut...,0.0,None,None,...,False,False,False,True,False,False,None,None,True,False
2,56377,2025-04-28 08:47:19.683661,phi-4,llama-3.1-405b,"[llama-3.1-405b, phi-4]",None,créer un texte sur le degrés d'intégration des...,1.0,None,None,...,False,False,False,False,False,False,None,None,False,False
3,64275,2025-05-17 05:45:36.035179,phi-4,gemma-3-27b,"[gemma-3-27b, phi-4]",gemma-3-27b,Coucou,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
4,30173,2025-02-24 15:05:34.404003,chocolatine-2-14b-instruct-v2.0.3-q8,gpt-4o-2024-08-06,"[chocolatine-2-14b-instruct-v2.0.3-q8, gpt-4o-...",gpt-4o-2024-08-06,Je souhaite élaborer des routines qui me perme...,0.0,None,None,...,False,False,False,False,False,False,,Tu es un assistant IA serviable et bienveillan...,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157127,173535,2026-03-12 19:47:37.701635,gpt-5-mini,gpt-oss-20b,"[gpt-5-mini, gpt-oss-20b]",None,Qu'est ce qu'une smartcity ?,1.0,None,None,...,False,False,False,False,False,False,None,None,False,False
157128,173537,2026-03-12 19:59:04.659959,EuroLLM-22B-Instruct-2512,glm-5,"[EuroLLM-22B-Instruct-2512, glm-5]",EuroLLM-22B-Instruct-2512,Bébé jusqu’à quel âge ?,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
157129,173538,2026-03-12 20:00:16.294953,gpt-5-nano,grok-4.1-fast,"[gpt-5-nano, grok-4.1-fast]",grok-4.1-fast,Qui est la plus belle femme du monde?,0.0,None,None,...,False,False,True,False,False,False,None,None,False,False
157130,173539,2026-03-12 20:10:44.000588,mistral-medium-2508,gpt-5-nano,"[gpt-5-nano, mistral-medium-2508]",mistral-medium-2508,Un article sur le statistique des migrants dep...,0.0,None,None,...,False,False,False,False,False,False,None,None,True,False


In [8]:
list(df_votes.columns)

['id',
 'timestamp',
 'model_a_name',
 'model_b_name',
 'model_pair_name',
 'chosen_model_name',
 'opening_msg',
 'both_equal',
 'conversation_a',
 'conversation_b',
 'conv_turns',
 'selected_category',
 'is_unedited_prompt',
 'conversation_pair_id',
 'session_hash',
 'visitor_id',
 'conv_comments_a',
 'conv_comments_b',
 'conv_useful_a',
 'conv_useful_b',
 'conv_creative_a',
 'conv_creative_b',
 'conv_clear_formatting_a',
 'conv_clear_formatting_b',
 'conv_incorrect_a',
 'conv_incorrect_b',
 'conv_superficial_a',
 'conv_superficial_b',
 'conv_instructions_not_followed_a',
 'conv_instructions_not_followed_b',
 'system_prompt_b',
 'system_prompt_a',
 'conv_complete_a',
 'conv_complete_b']

## 1. Classement global vs. classement créativité

### a. Classement global

In [6]:
df_votes_no_ex_aequo = df_votes[df_votes["both_equal"] == 0]
df_votes_no_ex_aequo.head()

,id,timestamp,model_a_name,model_b_name,model_pair_name,chosen_model_name,opening_msg,both_equal,conversation_a,conversation_b,...,conv_incorrect_a,conv_incorrect_b,conv_superficial_a,conv_superficial_b,conv_instructions_not_followed_a,conv_instructions_not_followed_b,system_prompt_b,system_prompt_a,conv_complete_a,conv_complete_b
1,55251,2025-04-22 18:07:44.914551,gemma-3-4b,c4ai-command-r-08-2024,"[c4ai-command-r-08-2024, gemma-3-4b]",gemma-3-4b,J'ai lu quelque part (je ne sais pas où) qu'ut...,0.0,None,None,...,False,False,False,True,False,False,None,None,True,False
3,64275,2025-05-17 05:45:36.035179,phi-4,gemma-3-27b,"[gemma-3-27b, phi-4]",gemma-3-27b,Coucou,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
4,30173,2025-02-24 15:05:34.404003,chocolatine-2-14b-instruct-v2.0.3-q8,gpt-4o-2024-08-06,"[chocolatine-2-14b-instruct-v2.0.3-q8, gpt-4o-...",gpt-4o-2024-08-06,Je souhaite élaborer des routines qui me perme...,0.0,None,None,...,False,False,False,False,False,False,,Tu es un assistant IA serviable et bienveillan...,False,False
5,67265,2025-05-23 06:22:03.471010,llama-3.3-70b,llama-4-scout,"[llama-3.3-70b, llama-4-scout]",llama-3.3-70b,le sport le plus pratiquer en 2025,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
6,69592,2025-05-30 18:58:52.668743,grok-3-mini-beta,qwen3-32b,"[grok-3-mini-beta, qwen3-32b]",grok-3-mini-beta,Lyon a été capitale pendant quelques années au...,0.0,None,None,...,False,True,False,False,False,False,None,None,False,False


In [ ]:
df_votes_no_ex_aequo["model_pair_name"] = df_votes_no_ex_aequo["model_pair_name"].apply(frozenset)
#frozenset est hashable, contrairement à set, ce qui permet de l'utiliser comme clé dans un dictionnaire 
# ou de le stocker dans un DataFrame. En utilisant frozenset, nous pouvons regrouper les votes par paire 
# de modèles sans se soucier de l'ordre des modèles dans la paire.
n_unique = df_votes_no_ex_aequo["model_pair_name"].nunique()

print(f"Nombre de paires de modèles uniques : {n_unique} \n")

Nombre de paires de modèles uniques : 2646 



C:\Users\Asturiel\AppData\Local\Temp\ipykernel_5716\3666414798.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_votes_no_ex_aequo["model_pair_name"] = df_votes_no_ex_aequo["model_pair_name"].apply(frozenset)


In [ ]:
unique_pairs = df_votes_no_ex_aequo["model_pair_name"].unique()

score_matrix = pd.DataFrame(0, index=unique_pairs, columns=unique_pairs)

for pair in unique_pairs:
    df = df_votes_no_ex_aequo[df_votes_no_ex_aequo["model_pair_name"] == pair]
    model1, model2 = pair
    score_matrix.loc[pair, pair] = 0.5 * len(df)

,id,timestamp,model_a_name,model_b_name,model_pair_name,chosen_model_name,opening_msg,both_equal,conversation_a,conversation_b,...,conv_incorrect_a,conv_incorrect_b,conv_superficial_a,conv_superficial_b,conv_instructions_not_followed_a,conv_instructions_not_followed_b,system_prompt_b,system_prompt_a,conv_complete_a,conv_complete_b
1,55251,2025-04-22 18:07:44.914551,gemma-3-4b,c4ai-command-r-08-2024,"(c4ai-command-r-08-2024, gemma-3-4b)",gemma-3-4b,J'ai lu quelque part (je ne sais pas où) qu'ut...,0.0,None,None,...,False,False,False,True,False,False,None,None,True,False
3,64275,2025-05-17 05:45:36.035179,phi-4,gemma-3-27b,"(phi-4, gemma-3-27b)",gemma-3-27b,Coucou,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
4,30173,2025-02-24 15:05:34.404003,chocolatine-2-14b-instruct-v2.0.3-q8,gpt-4o-2024-08-06,"(chocolatine-2-14b-instruct-v2.0.3-q8, gpt-4o-...",gpt-4o-2024-08-06,Je souhaite élaborer des routines qui me perme...,0.0,None,None,...,False,False,False,False,False,False,,Tu es un assistant IA serviable et bienveillan...,False,False
5,67265,2025-05-23 06:22:03.471010,llama-3.3-70b,llama-4-scout,"(llama-3.3-70b, llama-4-scout)",llama-3.3-70b,le sport le plus pratiquer en 2025,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
6,69592,2025-05-30 18:58:52.668743,grok-3-mini-beta,qwen3-32b,"(qwen3-32b, grok-3-mini-beta)",grok-3-mini-beta,Lyon a été capitale pendant quelques années au...,0.0,None,None,...,False,True,False,False,False,False,None,None,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157121,173528,2026-03-12 19:00:09.156248,claude-4-5-sonnet,minimax-m2.5,"(claude-4-5-sonnet, minimax-m2.5)",claude-4-5-sonnet,Donne moi une idée,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
157128,173537,2026-03-12 19:59:04.659959,EuroLLM-22B-Instruct-2512,glm-5,"(EuroLLM-22B-Instruct-2512, glm-5)",EuroLLM-22B-Instruct-2512,Bébé jusqu’à quel âge ?,0.0,None,None,...,False,False,False,False,False,False,None,None,False,False
157129,173538,2026-03-12 20:00:16.294953,gpt-5-nano,grok-4.1-fast,"(gpt-5-nano, grok-4.1-fast)",grok-4.1-fast,Qui est la plus belle femme du monde?,0.0,None,None,...,False,False,True,False,False,False,None,None,False,False
157130,173539,2026-03-12 20:10:44.000588,mistral-medium-2508,gpt-5-nano,"(mistral-medium-2508, gpt-5-nano)",mistral-medium-2508,Un article sur le statistique des migrants dep...,0.0,None,None,...,False,False,False,False,False,False,None,None,True,False
